In [13]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/spotify_artist_streaming_2020_2025.csv")
df.head()

,track_id,track_name,artist_name,album_name,release_date,genre,duration_ms,popularity,danceability,energy,...,popularity_category,loudness_category,key_name,mode_name,is_explicit_bool,release_quarter,is_weekend_release,log_stream_count,upbeat_score,artist_track_count
0,346d85e2-ffde-41b3-854a-589ed880b91f,Storm,Ethereal,The Hollow Journeys,2021-10-06,Pop,169594,47,0.6777,0.7949,...,Medium,Loud,A,Major,False,Q4,False,10.7492,0.6632,3232
1,12bc268a-363b-4f19-8fa8-0ced7ae1e462,Silent,The Wolfs,Ancient Thunder,2024-10-26,K-Pop,256755,26,0.9405,0.7163,...,Medium,Moderate,C#,Major,False,Q4,True,8.5895,0.7615,20
2,30d627d9-878d-4bb3-be59-71feb6ed4f3f,Shattered Lines,LolaOre,Twisted Harbors,2022-11-12,Punk,170212,46,0.6853,0.9500,...,Medium,Loud,E,Major,False,Q4,True,12.2935,0.7828,199
3,85501bb9-d221-4339-9e2e-29b1b3daff0a,Breaking Roses,Clay Black,The Blinding Letters,2020-06-22,Pop,181250,4,0.7817,0.6943,...,Low,Moderate,F#,Major,False,Q2,False,8.5658,0.6778,693
4,7bf92bd2-c6fe-4869-a90e-242bb8bbe7a3,Lines,Knot Hart,The Cosmic Letters,2024-01-22,Latin,160894,27,0.7200,0.4958,...,Medium,Moderate,C#,Major,False,Q1,False,10.2191,0.5494,12111


In [3]:
df["release_date"] = pd.to_datetime(df["release_date"])
df["release_date"].dtype

dtype('<M8[us]')

In [4]:
df[[
    "release_date",
    "release_year",
    "release_month",
    "release_day_of_week",
    "release_quarter",
    "is_weekend_release"
]].head(10)

,release_date,release_year,release_month,release_day_of_week,release_quarter,is_weekend_release
0,2021-10-06,2021,10,Wednesday,Q4,False
1,2024-10-26,2024,10,Saturday,Q4,True
2,2022-11-12,2022,11,Saturday,Q4,True
3,2020-06-22,2020,6,Monday,Q2,False
4,2024-01-22,2024,1,Monday,Q1,False
5,2021-04-12,2021,4,Monday,Q2,False
6,2025-07-16,2025,7,Wednesday,Q3,False
7,2023-12-12,2023,12,Tuesday,Q4,False
8,2020-06-17,2020,6,Wednesday,Q2,False
9,2020-06-09,2020,6,Tuesday,Q2,False


In [5]:
validacion_fechas = (
    (df["release_year"] == df["release_date"].dt.year) &
    (df["release_month"] == df["release_date"].dt.month) &
    (df["release_day_of_week"] == df["release_date"].dt.day_name()) &
    (df["release_quarter"] == "Q" + df["release_date"].dt.quarter.astype(str)) &
    (df["is_weekend_release"] == (df["release_date"].dt.dayofweek >= 5))
)

validacion_fechas.value_counts()

True    50000
Name: count, dtype: int64

In [6]:
duracion_calculada = df["duration_ms"] / 60000
(df["duration_minutes"] - duracion_calculada).abs().max()

np.float64(0.0050000000000003375)

In [7]:
df[["duration_ms", "duration_minutes"]].head(10)

,duration_ms,duration_minutes
0,169594,2.83
1,256755,4.28
2,170212,2.84
3,181250,3.02
4,160894,2.68
5,198179,3.30
6,251330,4.19
7,208198,3.47
8,254751,4.25
9,310076,5.17


In [8]:
mode_esperado = df["mode"].map({0: "Minor", 1: "Major"})
(df["mode_name"] == mode_esperado).value_counts()

True    50000
Name: count, dtype: int64

In [9]:
print("key_name:")
print(df[["key", "key_name"]].drop_duplicates().sort_values("key"))

print("\nis_explicit_bool:")
print((df["explicit"] == df["is_explicit_bool"]).value_counts())

print("\nartist_track_count:")
artist_count_esperado = df.groupby("artist_name")["track_id"].transform("count")
print((df["artist_track_count"] == artist_count_esperado).value_counts())

key_name:
    key key_name
14    0        C
1     1       C#
19    2        D
9     3       D#
2     4        E
11    5        F
3     6       F#
6     7        G
5     8       G#
0     9        A
10   10       A#
24   11        B

is_explicit_bool:
True    50000
Name: count, dtype: int64

artist_track_count:
True    50000
Name: count, dtype: int64


In [10]:
print("popularity_category:")
print(df["popularity_category"].value_counts())

print("\nloudness_category:")
print(df["loudness_category"].value_counts())

popularity_category:
popularity_category
Low          23268
Medium       21304
High          5181
Very High      247
Name: count, dtype: int64

loudness_category:
loudness_category
Moderate    34309
Loud        13240
Quiet        2451
Name: count, dtype: int64


In [11]:
print("Rangos de popularity:")
print(df.groupby("popularity_category")["popularity"].agg(["min", "max"]))

print("\nRangos de loudness:")
print(df.groupby("loudness_category")["loudness"].agg(["min", "max"]))

Rangos de popularity:
                     min  max
popularity_category          
High                  50   74
Low                    0   24
Medium                25   49
Very High             75   92

Rangos de loudness:
                     min    max
loudness_category              
Loud               -6.00   0.00
Moderate          -12.00  -6.01
Quiet             -18.62 -12.01


In [12]:
df[["stream_count", "log_stream_count"]].head(10)

,stream_count,log_stream_count
0,46592,10.7492
1,5374,8.5895
2,218262,12.2935
3,5248,8.5658
4,27420,10.2191
5,18568,9.8292
6,14871,9.6072
7,1068,6.9745
8,4146,8.3301
9,183936,12.1223


In [14]:
log_calculado = np.log1p(df["stream_count"])
(df["log_stream_count"] - log_calculado).abs().max()

np.float64(4.9998570167630874e-05)

In [15]:
df[["danceability", "energy", "upbeat_score"]].head(10)

,danceability,energy,upbeat_score
0,0.6777,0.7949,0.6632
1,0.9405,0.7163,0.7615
2,0.6853,0.9500,0.7828
3,0.7817,0.6943,0.6778
4,0.7200,0.4958,0.5494
5,0.6538,0.3732,0.4586
6,0.7399,0.4000,0.5475
7,0.1730,0.3085,0.2482
8,0.6838,0.5026,0.5249
9,0.5759,0.8349,0.6488


In [16]:
upbeat_calculado = (
    df["danceability"] * 0.6 +
    df["energy"] * 0.4
)

(df["upbeat_score"] - upbeat_calculado).abs().max()

np.float64(0.19297999999999993)

In [17]:
upbeat_calculado = (
    df["danceability"] + df["energy"]
) / 2

(df["upbeat_score"] - upbeat_calculado).abs().max()

np.float64(0.17445)

In [18]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
print("Valores nulos:", df.isnull().sum().sum())
print("Filas duplicadas:", df.duplicated().sum())
print("Track ID duplicados:", df["track_id"].duplicated().sum())

Filas: 50000
Columnas: 33
Valores nulos: 0
Filas duplicadas: 0
Track ID duplicados: 0


In [19]:
df.to_csv(
    "../data/processed/spotify_streaming_clean.csv",
    index=False
)

In [20]:
df_clean = pd.read_csv("../data/processed/spotify_streaming_clean.csv")

print("Filas:", df_clean.shape[0])
print("Columnas:", df_clean.shape[1])
print("Valores nulos:", df_clean.isnull().sum().sum())
print("Duplicados:", df_clean.duplicated().sum())

Filas: 50000
Columnas: 33
Valores nulos: 0
Duplicados: 0
